<a href="https://www.kaggle.com/code/avikdas567/birdclef-2026-mel-spectrogram-cnn-soundscapes?scriptVersionId=303071519" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import librosa
from tqdm.auto import tqdm

# Config

DATA_PATH = "/kaggle/input/competitions/birdclef-2026"

TRAIN_AUDIO = f"{DATA_PATH}/train_audio"
TRAIN_SOUNDSCAPES = f"{DATA_PATH}/train_soundscapes"
TEST_AUDIO = f"{DATA_PATH}/test_soundscapes"

SR = 32000
DURATION = 5
SAMPLES = SR * DURATION

N_MELS = 128
FMIN = 20
FMAX = 16000

N_FFT = 1024
HOP = 320

BATCH_SIZE = 64
EPOCHS = 1
LR = 1e-3

DEVICE = "cpu"
SEED = 42

MAX_AUDIO_TRAIN = 2500
MAX_SCAPE_TRAIN = 2500

# Seed

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(SEED)

# Load Metadata

train = pd.read_csv(f"{DATA_PATH}/train.csv")
taxonomy = pd.read_csv(f"{DATA_PATH}/taxonomy.csv")
scape_labels = pd.read_csv(f"{DATA_PATH}/train_soundscapes_labels.csv")

species = taxonomy.primary_label.values
NUM_CLASSES = len(species)

species_to_idx = {s:i for i,s in enumerate(species)}

print("Classes:", NUM_CLASSES)

# Convert HH:MM:SS timestamps → seconds

def time_to_seconds(t):

    try:
        parts = str(t).split(":")
        parts = [float(p) for p in parts]

        if len(parts) == 3:
            return parts[0]*3600 + parts[1]*60 + parts[2]

        if len(parts) == 2:
            return parts[0]*60 + parts[1]

        return float(parts[0])

    except:
        return None

# Precompute Mel Filter

mel_filter = librosa.filters.mel(
    sr=SR,
    n_fft=N_FFT,
    n_mels=N_MELS,
    fmin=FMIN,
    fmax=FMAX
)

# Faster Feature Extraction

def extract_features(audio):

    spec = np.abs(
        librosa.stft(
            audio,
            n_fft=N_FFT,
            hop_length=HOP
        )
    )**2

    mel = np.dot(mel_filter, spec)

    mel = librosa.power_to_db(mel)

    delta = librosa.feature.delta(mel)

    feat = np.stack([mel, delta], axis=0)

    feat = (feat - feat.mean())/(feat.std()+1e-6)

    return feat.astype(np.float32)

# Build Training Dataset

X = []
Y = []

# train_audio
print("\nLoading train_audio...")

train_small = train.sample(MAX_AUDIO_TRAIN, random_state=SEED)

for _,row in tqdm(train_small.iterrows(),
                  total=len(train_small),
                  desc="train_audio"):

    path = os.path.join(TRAIN_AUDIO,row.filename)

    y,_ = librosa.load(path, sr=SR, mono=True)

    if len(y) < SAMPLES:
        y = np.pad(y,(0,SAMPLES-len(y)))

    y = y[:SAMPLES]

    feat = extract_features(y)

    label = np.zeros(NUM_CLASSES)

    label[species_to_idx[row.primary_label]] = 1

    X.append(feat)
    Y.append(label)

# train_soundscapes
print("\nLoading soundscape labels...")

scape_small = scape_labels.sample(
    min(MAX_SCAPE_TRAIN,len(scape_labels)),
    random_state=SEED
)

for _,row in tqdm(scape_small.iterrows(),
                  total=len(scape_small),
                  desc="soundscapes"):

    path = os.path.join(TRAIN_SOUNDSCAPES,row.filename)

    if not os.path.exists(path):
        continue

    start_sec = time_to_seconds(row.start)
    end_sec = time_to_seconds(row.end)

    if start_sec is None or end_sec is None:
        continue

    audio,_ = librosa.load(path, sr=SR, mono=True)

    start = int(start_sec * SR)
    end = int(end_sec * SR)

    clip = audio[start:end]

    if len(clip) < SAMPLES:
        clip = np.pad(clip,(0,SAMPLES-len(clip)))

    clip = clip[:SAMPLES]

    feat = extract_features(clip)

    label = np.zeros(NUM_CLASSES)

    for s in str(row.primary_label).split(";"):
        if s in species_to_idx:
            label[species_to_idx[s]] = 1

    X.append(feat)
    Y.append(label)

X = np.stack(X)
Y = np.stack(Y)

print("\nTraining samples:",len(X))

# Dataset

class BirdDataset(Dataset):

    def __init__(self,X,Y):
        self.X = torch.tensor(X)
        self.Y = torch.tensor(Y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        return self.X[idx],self.Y[idx]

dataset = BirdDataset(X,Y)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Model

class FastCNN(nn.Module):

    def __init__(self,n_classes):

        super().__init__()

        self.conv1 = nn.Conv2d(2,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.conv3 = nn.Conv2d(64,128,3,padding=1)

        self.pool = nn.MaxPool2d(2)

        self.gap = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Linear(128,n_classes)

    def forward(self,x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.gap(x)

        x = torch.flatten(x,1)

        return self.fc(x)

model = FastCNN(NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(),lr=LR)

criterion = nn.BCEWithLogitsLoss()

# Train

print("\nTraining...")

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for x,y in tqdm(loader, desc="training"):

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        pred = model(x)

        loss = criterion(pred,y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Loss:",total_loss/len(loader))

# Inference

print("\nRunning inference...")

model.eval()

rows = []

test_files = [f for f in os.listdir(TEST_AUDIO) if f.endswith(".ogg")]

for fname in tqdm(test_files, desc="test_soundscapes"):

    path = os.path.join(TEST_AUDIO,fname)

    audio,_ = librosa.load(path,sr=SR, mono=True)

    total_segments = len(audio)//SAMPLES

    for i in range(total_segments):

        start = i*SAMPLES
        end = start+SAMPLES

        clip = audio[start:end]

        if len(clip)<SAMPLES:
            clip = np.pad(clip,(0,SAMPLES-len(clip)))

        feat = extract_features(clip)

        x = torch.tensor(feat).unsqueeze(0)

        with torch.no_grad():

            pred = model(x)

            pred = torch.sigmoid(pred).numpy()[0]

        pred = pred ** 0.8

        row_id = fname.replace(".ogg","") + f"_{(i+1)*5}"

        row = {"row_id":row_id}

        for s,p in zip(species,pred):
            row[s] = p

        rows.append(row)

submission = pd.DataFrame(rows)

sample = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")
submission = submission.reindex(columns=sample.columns, fill_value=0)

submission.to_csv("submission.csv", index=False)

print("\nSubmission saved!")
print(submission.head())

Classes: 234

Loading train_audio...


train_audio:   0%|          | 0/2500 [00:00<?, ?it/s]


Loading soundscape labels...


soundscapes:   0%|          | 0/1478 [00:00<?, ?it/s]


Training samples: 3978

Training...


training:   0%|          | 0/63 [00:00<?, ?it/s]

Loss: 0.14084789423005922

Running inference...


test_soundscapes: 0it [00:00, ?it/s]


Submission saved!
Empty DataFrame
Columns: [row_id, 1161364, 116570, 1176823, 1491113, 1595929, 209233, 22930, 22956, 22961, 22967, 22973, 22983, 22985, 23150, 23154, 23158, 23176, 23724, 24279, 24285, 24287, 24321, 244024, 25073, 25092, 25214, 326272, 41970, 43435, 47144, 47158son01, 47158son02, 47158son03, 47158son04, 47158son05, 47158son06, 47158son07, 47158son08, 47158son09, 47158son10, 47158son11, 47158son12, 47158son13, 47158son14, 47158son15, 47158son16, 47158son17, 47158son18, 47158son19, 47158son20, 47158son21, 47158son22, 47158son23, 47158son24, 47158son25, 476521, 516975, 517063, 555123, 555145, 555146, 64898, 65377, 65380, 66971, 67107, 67252, 70711, 738183, 74113, 74580, 760266, ashgre1, astcra1, bafcur1, baffal1, banana, barant1, batbel1, baymac, bbwduc, bcwfin2, bkcdon, bkhpar, blchaw1, blheag1, blttit1, bncfly, bobfly1, brcmar1, brnowl, bucmot4, bucpar, bufpar, bunibi1, burowl, camfli1, chacha1, chbmoc1, ...]
Index: []

[0 rows x 235 columns]
